# 机械工程小助手 — LoRA 微调（Kaggle 免费 GPU）

## 使用步骤
1. 打开 https://www.kaggle.com 注册/登录
2. 右上角 **Create → Notebook**
3. 右侧 **Settings** 面板里设置两项：
   - **Accelerator** → `GPU T4 x2`
   - **Internet** → `On`（必须开，否则下载不了模型）
4. 把本文件上传到 Kaggle（File → Import Notebook），或直接复制每个 cell
5. 直接 **Run All**（数据已内置 30 条机械工程问答）

## 这个 notebook 干了什么
| 步骤 | 说明 |
|---|---|
| 加载模型 | Qwen2.5-1.5B-Instruct，fp16 放显存 |
| 构造数据 | 按 chat template 拼成 token，**只在 assistant 回答部分算 loss** |
| 挂 LoRA | 只训练 <1% 的参数（约 900 万），冻结原模型 |
| 训练 | 10 个 epoch，约 60 步 |
| 对比 | 同一个问题，关掉 adapter 和打开 adapter 各答一次 |
| 保存 | 导出 adapter（约 20MB），可下载回本地 |


In [ ]:
# ===== 1. 环境检查 + 装依赖 =====
!nvidia-smi

# Kaggle 自带 torch / transformers，只需补 peft
!pip install -q peft datasets accelerate

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('CUDA 可用:', torch.cuda.is_available())


In [ ]:
# ===== 2. 训练数据：30 条机械工程问答 =====
# 覆盖：机械设计、机械原理、公差配合、材料热处理、液压气动、PLC/传感器

DATA = [
    {'messages': [
        {'role': 'user', 'content': '键连接的作用是什么？'},
        {'role': 'assistant', 'content': '键连接用于实现轴与轮毂之间的周向固定，并传递扭矩。常见类型有平键、半圆键和楔键，其中平键装拆方便、对中性好，应用最广。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': 'V带传动为什么比平带传动承载能力高？'},
        {'role': 'assistant', 'content': 'V带靠两侧面与轮槽之间的楔形摩擦传力，当量摩擦系数约为平带的 3 倍。因此在同样张紧力下，V带不易打滑，承载能力更高。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '轴承代号 6208 表示什么意思？'},
        {'role': 'assistant', 'content': '6 表示深沟球轴承，2 表示尺寸系列为轻系列，08 表示内径为 8×5=40 mm。这是最常用的一类滚动轴承代号。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '齿轮传动和带传动相比各有什么优缺点？'},
        {'role': 'assistant', 'content': '齿轮传动传动比准确、效率高、结构紧凑、承载能力大、寿命长；但制造和安装精度要求高、成本高、不适合远距离传动、没有过载保护。带传动能缓冲吸振、过载时打滑起保护作用、适合中心距较大的场合，但传动比不准确、寿命较短。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '标准直齿圆柱齿轮为什么齿数一般不少于 17？'},
        {'role': 'assistant', 'content': '用范成法加工标准齿轮时，齿数少于 17 齿会发生根切，把齿根部分的渐开线切掉，削弱齿根强度并影响传动平稳性。这一限制对应标准压力角 20 度、齿顶高系数 1 的情况。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '斜齿轮和直齿轮相比有什么特点？'},
        {'role': 'assistant', 'content': '斜齿轮重合度大，传动更平稳、噪声更小、承载能力更高；缺点是啮合时会产生轴向力，需要能承受轴向载荷的轴承。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '滚动轴承和滑动轴承怎么选？'},
        {'role': 'assistant', 'content': '滚动轴承摩擦小、启动灵活、已标准化、互换性好，适合中低速和一般载荷；滑动轴承在高速、重载、有冲击或要求特别平稳低噪声的场合更合适，形成液体动压润滑后寿命很长。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '联轴器和离合器有什么区别？'},
        {'role': 'assistant', 'content': '联轴器用来把两根轴连接起来并传递扭矩，工作时两轴不能分离，只有停机拆开才能分开；离合器可以在机器运转过程中随时接合或分离。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '螺纹连接为什么需要防松？常用方法有哪些？'},
        {'role': 'assistant', 'content': '螺纹升角小于当量摩擦角时虽能自锁，但在振动和冲击下仍会逐渐松动。常用防松方法有三类：摩擦防松（双螺母、弹簧垫圈、自锁螺母）、机械防松（开口销、止动垫圈、串联钢丝）、永久防松（点焊、粘接）。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '螺栓预紧力有什么作用？'},
        {'role': 'assistant', 'content': '预紧力提高连接的紧密性和可靠性，使螺栓在工作载荷作用下的应力变化幅度减小，从而提高疲劳强度；对有密封要求的连接，预紧力还是保证密封的必要条件。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '平键的尺寸怎么确定？'},
        {'role': 'assistant', 'content': '键的宽度 b 和高度 h 按轴的直径从标准中查取，键的长度 L 按轮毂长度确定，一般比轮毂略短，并要满足标准长度系列。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '公差代号 H7/g6 表示什么配合？'},
        {'role': 'assistant', 'content': '表示基孔制间隙配合。孔的公差等级为 IT7，轴为 IT6，g 是轴的基本偏差代号，上偏差为很小的负值。这种配合间隙小，用于要求定位精确又能相对滑动的场合，例如滑动轴承与轴颈。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '为什么优先采用基孔制？'},
        {'role': 'assistant', 'content': '因为加工孔比加工轴困难，孔的尺寸受钻头、铰刀、拉刀等定尺寸刀具限制，规格有限；把孔的公差带固定为 H，通过改变轴的公差带来得到各种配合，可以减少刀具和量具的种类，更经济。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '形状公差和位置公差各包括哪些项目？'},
        {'role': 'assistant', 'content': '形状公差有直线度、平面度、圆度、圆柱度，另外有线轮廓度和面轮廓度。位置公差分三类：定向公差（平行度、垂直度、倾斜度）、定位公差（同轴度、对称度、位置度）、跳动公差（圆跳动、全跳动）。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '表面粗糙度 Ra 是什么含义？'},
        {'role': 'assistant', 'content': 'Ra 是轮廓算术平均偏差，指在取样长度内轮廓偏距绝对值的算术平均值，单位是微米。Ra 越小表面越光滑，加工成本也越高。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '45 钢的调质处理是什么？'},
        {'role': 'assistant', 'content': '调质是淬火加高温回火，回火温度约 500 到 650 度，处理后得到回火索氏体组织，强度和韧性兼顾，综合力学性能好，常用于轴类、连杆、齿轮等零件。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '淬火后为什么必须回火？'},
        {'role': 'assistant', 'content': '淬火得到马氏体，硬度高但很脆、内应力大，直接使用容易开裂。回火可以消除内应力、降低脆性并调整性能：低温回火保持硬度，中温回火获得弹性，高温回火获得良好的综合力学性能。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '退火、正火、淬火、回火有什么区别？'},
        {'role': 'assistant', 'content': '退火加热后缓慢冷却，降低硬度便于切削、消除内应力、细化晶粒；正火冷却速度快于退火，强度和硬度略高，常作预备热处理；淬火快速冷却获得马氏体，提高硬度和耐磨性；回火是淬火后加热保温再冷却，消除应力并调整性能。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '灰铸铁和球墨铸铁有什么区别？'},
        {'role': 'assistant', 'content': '区别在于石墨形态。灰铸铁是片状石墨，抗拉强度和塑性低，但减振性好、铸造性能好、价格低；球墨铸铁是球状石墨，强度和韧性接近钢，可用来做曲轴、齿轮等受力零件。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '平面机构的自由度怎么计算？'},
        {'role': 'assistant', 'content': '用公式 F 等于 3n 减去 2 倍的 PL 再减去 PH，其中 n 是活动构件数，PL 是低副数目，PH 是高副数目。计算前要先去掉局部自由度和虚约束。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '铰链四杆机构曲柄存在的条件是什么？'},
        {'role': 'assistant', 'content': '有两个条件：一是最短杆与最长杆长度之和小于或等于其余两杆长度之和；二是最短杆为机架或与机架相连的连架杆。两个条件同时满足才有曲柄。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '凸轮从动件常用哪些运动规律？'},
        {'role': 'assistant', 'content': '常用等速运动、等加速等减速运动、余弦加速度即简谐运动和正弦加速度即摆线运动。等速运动在起止点加速度无穷大，产生刚性冲击；后两种冲击小，适合较高转速。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '为什么高速凸轮要采用修正运动规律？'},
        {'role': 'assistant', 'content': '高速时惯性力很大，等速规律在行程起止点加速度突变会产生刚性冲击，等加速规律会产生柔性冲击。所以高速凸轮采用正弦加速度等无冲击或冲击很小的运动规律。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '齿轮的模数是什么？'},
        {'role': 'assistant', 'content': '模数 m 等于分度圆直径除以齿数，即 m 等于 d 除以 z，单位是毫米。它是齿轮尺寸的基本参数，模数越大齿越大、弯曲强度越高、承载能力越强。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '齿轮标准压力角为什么取 20 度？'},
        {'role': 'assistant', 'content': '压力角越大齿根越厚、强度越高，但传力的有效分量越小、效率下降，还容易自锁。20 度是在强度、传动性能和加工工艺之间折中的标准值。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '液压系统中溢流阀有什么作用？'},
        {'role': 'assistant', 'content': '在定量泵系统中溢流阀常开，把多余的流量排回油箱，同时使系统压力保持恒定，起调压作用；在变量泵系统或作安全阀使用时，它限制系统最高压力，防止过载。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '液压传动和气压传动有什么区别？'},
        {'role': 'assistant', 'content': '液压油几乎不可压缩，能输出很大推力，运动平稳、定位精度高，但需要回油管路，存在泄漏和污染；空气可压缩，动作快、成本低、清洁，但输出力小、定位精度差、噪声大。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': 'PLC 的一个扫描周期包括哪几个阶段？'},
        {'role': 'assistant', 'content': '包括输入采样、用户程序执行、输出刷新三个阶段，然后不断循环。扫描周期越长响应越慢，所以程序要尽量精简。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '增量式编码器和绝对式编码器有什么区别？'},
        {'role': 'assistant', 'content': '增量式编码器输出脉冲，通过计数得到位移，断电后位置信息丢失，上电需要回零；绝对式编码器每个位置有唯一编码，断电后位置仍能保持，不需要回零。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '步进电机和伺服电机有什么区别？'},
        {'role': 'assistant', 'content': '步进电机是开环控制，成本低、低速扭矩大，但高速容易失步；伺服电机是闭环控制，带编码器反馈，精度高、响应快、过载能力强，但价格高、调试复杂。'},
    ]},
]

print(f'训练样本数：{len(DATA)} 条')
assert len(DATA) >= 20, '至少 20 条再训练，否则模型学不到东西'


In [ ]:
# ===== 3. 加载模型和分词器 =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.config.use_cache = False
print('参数量：', round(model.num_parameters() / 1e9, 2), 'B')
print('设备：', next(model.parameters()).device)


In [ ]:
# ===== 4. 构造训练集 =====
# 关键点：labels 里把「提问部分」设成 -100，表示不计算损失
# 这样模型只学「怎么回答」，不学「怎么提问」
from torch.utils.data import Dataset

MAX_LEN = 512


def encode(messages):
    prompt_ids = tok.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True)
    full_ids = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False)
    full_ids = full_ids[:MAX_LEN]
    n_prompt = min(len(prompt_ids), len(full_ids))
    labels = [-100] * n_prompt + full_ids[n_prompt:]
    return {'input_ids': full_ids, 'labels': labels}


class QADataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return encode(self.rows[i]['messages'])


def collate(batch):
    max_len = max(len(b['input_ids']) for b in batch)
    pad_id = tok.pad_token_id or tok.eos_token_id
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        input_ids.append(b['input_ids'] + [pad_id] * pad)
        labels.append(b['labels'] + [-100] * pad)
        attn.append([1] * len(b['input_ids']) + [0] * pad)
    return {
        'input_ids': torch.tensor(input_ids),
        'labels': torch.tensor(labels),
        'attention_mask': torch.tensor(attn),
    }


ds = QADataset(DATA)
sample = ds[0]
n_mask = sum(1 for x in sample['labels'] if x == -100)
print(f'第 1 条总长 {len(sample["input_ids"])}，其中被屏蔽 {n_mask} 个 token')
print('校验：屏蔽数应该 > 0，说明 prompt 部分没有参与 loss')


In [ ]:
# ===== 5. 挂 LoRA =====
# r 越大能学的内容越多，但也越容易过拟合；小数据集用 8~16 足够
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
# ===== 6. 训练 =====
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='/kaggle/working/ckpt',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=2,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
    optim='adamw_torch',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=collate,
)

trainer.train()


In [ ]:
# ===== 7. loss 曲线 =====
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if 'loss' in h]
steps = [h['step'] for h in hist]
losses = [h['loss'] for h in hist]

plt.figure(figsize=(7, 3))
plt.plot(steps, losses, marker='o', markersize=3)
plt.xlabel('step')
plt.ylabel('loss')
plt.title('LoRA 训练损失')
plt.grid(alpha=0.3)
plt.show()

print(f'起始 loss: {losses[0]:.4f} → 结束 loss: {losses[-1]:.4f}')
print('正常情况：整体下降，末尾有波动（数据少导致的正常现象）')


In [ ]:
# ===== 8. 微调前后对比 =====
# 用同一个问题，分别关掉 / 打开 adapter 生成
from contextlib import nullcontext


def ask(question, use_adapter=True, max_new_tokens=200):
    msgs = [{'role': 'user', 'content': question}]
    ids = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors='pt')
    ids = ids.to(model.device)
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with ctx, torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


test_qs = [
    '键连接的作用是什么？',
    'V带传动为什么比平带传动承载能力高？',
    '轴承代号 6208 表示什么意思？',
]

for q in test_qs:
    print('=' * 60)
    print('问：', q)
    print('\n[微调前] ', ask(q, use_adapter=False)[:300])
    print('\n[微调后] ', ask(q, use_adapter=True)[:300])
    print()


In [ ]:
# ===== 9. 保存 adapter =====
# 只有 adapter 权重（约 20MB），下载方便
SAVE_DIR = '/kaggle/working/lora_adapter'
model.save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)

import os
total = 0
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    total += size
    print(f'  {f:30s} {size:8.1f} MB')
print(f'合计 {total:.1f} MB')
print('\n在 Kaggle 右侧 Output 面板可以下载整个 lora_adapter 目录')


## 下一步（可选）

### 把 adapter 合并进原模型，导出成能用 Ollama 跑的形式
```python
merged = model.merge_and_unload()
merged.save_pretrained('/kaggle/working/merged', safe_serialization=True)
tok.save_pretrained('/kaggle/working/merged')
```
合并后约 3GB。要变成 Ollama 能吃的 GGUF，还需要 llama.cpp 的 `convert_hf_to_gguf.py` 转换（这一步放到第四阶段周 15 一起做）。

### 面试怎么讲这个项目
1. **为什么用 LoRA**：全参微调 1.5B 需要约 20GB 显存存梯度、优化器状态和激活；LoRA 只训练低秩旁路矩阵，可训练参数降到 1% 以下，T4 单卡就能跑。
2. **为什么 labels 要屏蔽 prompt**：SFT 是让模型学「给定问题怎么答」，不是学「怎么问」。不屏蔽的话模型会把提问部分也算进损失，浪费容量。
3. **r 和 alpha 怎么选**：r 是低秩矩阵的秩，决定旁路容量；alpha/r 是缩放系数，常用 alpha 等于 2r。数据少就用小 r，防过拟合。
4. **数据量为什么这么少**：这只是一个最小可跑通的 demo。真实项目通常需要几千到几万条，并且要做数据清洗、去重、质量过滤。
